# RAFT local pipeline: 100 synthetic support cases

Walk through **worker passes → final review → timeline embeddings → local retrieval → case graph → graph expansion**, using real OpenAI Agents SDK runs and real OpenAI embeddings.

This notebook loads `OPENAI_API_KEY` from your existing `.env`; it never embeds or prints credentials. Running it makes billable API calls. The dataset contains 100 tiny, hand-written synthetic support cases across five incident themes, including 10 information-only requests. It is for inspecting the pipeline, not measuring extraction or retrieval quality.

Run cells in order with a Python 3.11+ kernel. From the repository root, install if needed:
```bash
python -m pip install -e ".[notebook]" jupyterlab python-dotenv
```
The checked-in sample data lives in `data/synthetic_cases.jsonl`. Its short generator is in `synthetic_data.py` next to this notebook. The default output schemas/prompts here are demonstrations; replace them with your own later.

**Start small if desired:** set `CASE_LIMIT=5` below; use 100 for the complete walkthrough. The standalone component examples also process four sample cases so you can inspect each component directly.


In [ ]:
import json
import os
import sys
from copy import deepcopy
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv

REPO = next(
    root
    for base in [Path.cwd(), *Path.cwd().parents]
    for root in (base, base / "raft")
    if (root / "src" / "raft").is_dir()
)
DEMO = REPO / "examples" / "notebooks"
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(DEMO))
# Project imports follow repository path setup.
from agents import Agent, ModelSettings, OpenAIResponsesModel, RunConfig  # noqa: E402
from openai import AsyncOpenAI  # noqa: E402
from synthetic_data import make_cases  # noqa: E402

from raft import LocalPipeline, build_case_graph, embed_cases, run_cases  # noqa: E402
from raft.defaults import (  # noqa: E402
    REVIEWER_INSTRUCTIONS,
    WORKER_INSTRUCTIONS,
    CaseExtraction,
    CaseReview,
    case_to_text,
    state_to_text,
)
from raft.embedding.openai import OpenAIEmbeddings  # noqa: E402
from raft.tools import edit_state, query_case_sql  # noqa: E402

load_dotenv(REPO / ".env", override=False)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in the environment or repository .env file.")

def show(value):
    print(json.dumps(value, indent=2, default=lambda obj: obj.model_dump(mode="json")))


CASE_LIMIT = 100
SHOW_PROGRESS = True  # tqdm.auto chooses notebook widgets or terminal bars.
AGENT_MODEL = os.getenv("RAFT_AGENT_MODEL", "gpt-5.2")
EMBEDDING_MODEL = os.getenv("RAFT_EMBEDDING_MODEL", "text-embedding-3-small")
print("Live OpenAI calls enabled; key loaded without displaying it.")
print("Models:", AGENT_MODEL, "/", EMBEDDING_MODEL)


## 1. Inspect the local synthetic inputs

Each data point is one case, containing metadata and ordered artifacts. `kind` is a convenient label in this synthetic dataset; production artifacts can have arbitrary JSON fields. No answers are supplied separately to the agents: they must extract findings from the artifacts.

The supplied file contains exactly 100 cases. `make_cases(...)` reproduces them deterministically. No scikit-learn dependency or external dataset download is needed for task-shaped support-ticket text.


In [ ]:
DATA_PATH = DEMO / "data" / "synthetic_cases.jsonl"
cases = [json.loads(line) for line in DATA_PATH.read_text().splitlines()]
assert len(cases) == 100
assert cases == make_cases(100)
show({
    "cases": len(cases),
    "incident_cases": sum(len(case["artifacts"]) > 1 for case in cases),
    "information_requests": sum(len(case["artifacts"]) == 1 for case in cases),
    "first_case": cases[0],
    "information_request": cases[9],
})


## 2. Construct agents and embedding backends outside RAFT

The worker uses `edit_state` to update a case across passes. The required reviewer can query evidence/history, correct state with the same tool, and returns its own structured `CaseReview`. Filtering runs **after** review, and filtered cases retain their full fields.

One worker definition and one reviewer definition are reused across cases and passes. RAFT supplies isolated invocation context. A native `RunConfig` passes through to the SDK.

The clients use your existing OpenAI key. Optional `RAFT_AGENT_BASE_URL` and `RAFT_EMBEDDING_BASE_URL` environment variables select endpoints. Model names can be changed above. Keep agent and embedding clients open until the final cleanup cell.


In [ ]:
agent_client = AsyncOpenAI(
    base_url=os.getenv("RAFT_AGENT_BASE_URL"), max_retries=0,
)
embedding_client = AsyncOpenAI(
    base_url=os.getenv("RAFT_EMBEDDING_BASE_URL"), max_retries=0,
)
clients = [agent_client, embedding_client]
agent_model = OpenAIResponsesModel(model=AGENT_MODEL, openai_client=agent_client)
entry_backend = OpenAIEmbeddings(client=embedding_client, model=EMBEDDING_MODEL)
graph_backend = OpenAIEmbeddings(client=embedding_client, model=EMBEDDING_MODEL)

worker = Agent(
    name="Demo worker", model=agent_model,
    instructions=WORKER_INSTRUCTIONS + (
        "\nCompletion rule: finish_pass marks THIS BATCH complete, not the whole case. "
        "Set finish_pass=True after processing the supplied batch even when batch.is_last=False. "
        "Before replying with final text, ensure edit_state reports pass_finished=True. "
        "If your edit left it False, call edit_state again with patch_json='[]' and finish_pass=True."
    ),
    model_settings=ModelSettings(reasoning={"effort": "low"}, max_tokens=3000),
    tools=[query_case_sql, edit_state],
)
reviewer = Agent(
    name="Demo reviewer", model=agent_model,
    instructions=REVIEWER_INSTRUCTIONS + (
        "\nFor this walkthrough every input is synthetic. Judge eligibility by technical "
        "content, not the synthetic metadata flag. Information-only requests remain ineligible."
    ),
    model_settings=ModelSettings(reasoning={"effort": "low"}, max_tokens=3000),
    tools=[query_case_sql, edit_state],
    output_type=CaseReview,
)

extraction_options = dict(
    worker_agent=worker, reviewer_agent=reviewer,
    output_type=CaseExtraction,
    should_keep=lambda case: case.review.extractable,
    id_field="id", metadata_field="metadata", artifacts_field="artifacts",
    artifact_sort_field="sequence",
    max_batch_chars=300,       # Whole artifacts per worker pass; demonstrates multiple passes.
    max_query_chars=4000,      # Full serialized SQL result limit.
    concurrency=8,            # Active cases.
    agent_concurrency=8,      # Concurrent worker/reviewer invocations across those cases.
    rpm=300,                  # Agent-run starts per minute, not individual model requests.
    timeout=180, retries=1, max_turns=10,
    run_config=RunConfig(tracing_disabled=True),
)
embedding_options = dict(
    backend=entry_backend, state_to_text=state_to_text,
    batch_size=2, concurrency=4, rpm=300,
)
graph_options = dict(
    backend=graph_backend, case_to_text=case_to_text,
    batch_size=16, top_k=5, concurrency=2, rpm=300,
)
print("Reusable agents:", worker.name, "/", reviewer.name)


## 3. Run extraction as an independent component

Start with three incidents plus an information request. The worker receives only its current batch plus accumulated case state. It does not filter cases. The reviewer runs after the worker finishes all passes.

This standalone call returns data without creating a local persistent index. The pipeline below calls the same components internally.


In [ ]:
extraction_result = await run_cases(
    cases=[cases[0], cases[1], cases[2], cases[9]],
    show_progress=SHOW_PROGRESS, **extraction_options,
)
show(extraction_result["summary"])
show({"failures": extraction_result["failed_cases"]})
assert not extraction_result["failed_cases"], "Inspect failures before continuing."
first_case = extraction_result["extracted_cases"][0]
show({
    "id": first_case.id,
    "output": first_case.output,
    "review": first_case.review,
    "passes": first_case.execution["passes"],
    "agent_usage_by_model": first_case.execution["usage"],
    "filtered_ids": [case.id for case in extraction_result["filtered_cases"]],
})
assert first_case.execution["passes"] > 1


## 4. Inspect committed revisions and reviewer tool calls

A revision records a committed state and edits from a successful worker pass (or reviewer correction). During review, `query_case_sql` exposes `state_revisions`; workers cannot read that table. The ephemeral case SQLite connection closes after extraction, but the returned execution record retains revisions and tool traces for inspection or optional saving.

The reviewer chooses which checks are useful and can call `edit_state` to correct the final output before returning its structured assessment. Inspect the trace below to see what this particular live run actually did.


In [ ]:
show({"revision_count": len(first_case.execution["revisions"])})
show(first_case.execution["revisions"][-1])
# Tool traces can be large: inspect just the last invocation (the reviewer).
show(first_case.execution["tool_calls"][-1])


## 5. Embed timeline entries as an independent component

`state_to_text` returns one string per timeline entry. Current timeline embedding batches **within each case**; six entries with `batch_size=2` use three requests. Only complete successful case vectors are published.

Token usage is one aggregate per component invocation; there is no per-case embedding token usage. OpenAI returns `prompt_tokens` and `total_tokens`; these overlapping counters should not be added together. Agent usage remains separate, grouped by model within each case.


In [ ]:
embedding_result = await embed_cases(
    cases=extraction_result["extracted_cases"],
    show_progress=SHOW_PROGRESS, **embedding_options,
)
assert not embedding_result["failed_cases"]
first_embedded = embedding_result["embedded_cases"][0]
vector_record = first_embedded["embeddings"][0]
show({
    "summary": embedding_result["summary"],
    "embedding_usage": embedding_result["embedding_usage"],
    "embedding_requests": embedding_result["embedding_requests"],
    "case_record_keys": list(first_embedded),
    "entry_preview": {**vector_record, "embedding": vector_record["embedding"][:6]},
    "vector_length": len(vector_record["embedding"]),
})


## 6. Build a case graph independently

`case_to_text` creates one summary per case. Graph embedding can batch summaries **across cases**. Graph nodes are cases, while retrieval above embeds individual timeline entries.

The graph first finds directed neighbors with dense + BM25 rank fusion. Edge weights then use shared-neighbor overlap; they are not cosine similarities. This tiny example is for inspecting structure, not assessing graph quality.


In [ ]:
small_graph_options = {**graph_options, "top_k": 2}
graph_result = await build_case_graph(
    cases=extraction_result["extracted_cases"],
    show_progress=SHOW_PROGRESS, **small_graph_options,
)
assert not graph_result["failed_cases"]
show({
    "summary": graph_result["summary"],
    "embedding_usage": graph_result["embedding_usage"],
    "embedding_requests": graph_result["embedding_requests"],
    "first_neighbor_group": graph_result["neighbors"][0],
    "first_edge": graph_result["edges"][0] if graph_result["edges"] else None,
})


## 7. Let the local pipeline orchestrate the components

With `CASE_LIMIT=100`, start with 80 cases and add the remaining 20 below. There are eight information-only requests in the first 80 and two in the final 20. The live reviewer decides eligibility; inspect the actual counts and any failures.

Each Run All creates a new directory under `outputs/notebook-demo/`; no existing output is deleted. Keep `OUTPUT_DIR` to reopen a particular run. `show_progress` propagates to component stages, using `tqdm.auto` for notebooks or terminals.

To resume an interrupted run, set `RAFT_NOTEBOOK_OUTPUT_DIR` to that run directory before starting. Saved outputs in this notebook include such a resume: successful cases were reused, so the first extraction summary counts only remaining work. Without that environment variable, Run All creates a fresh index.


In [ ]:
selected_cases = cases[:CASE_LIMIT]
split = max(1, int(len(selected_cases) * 0.8))
initial_cases, incoming_cases = selected_cases[:split], selected_cases[split:]
OUTPUT_DIR = Path(os.getenv(
    "RAFT_NOTEBOOK_OUTPUT_DIR",
    str(REPO / "outputs" / "notebook-demo" / ("live-" + uuid4().hex[:8])),
))

def open_pipeline(directory):
    return LocalPipeline(
        output_dir=directory, extraction=extraction_options,
        embedding=embedding_options, graph=graph_options,
        bm25=True, show_progress=SHOW_PROGRESS,
    )

pipeline = open_pipeline(OUTPUT_DIR)
index_result = await pipeline.index(initial_cases)
show({
    "output_dir": str(OUTPUT_DIR.relative_to(REPO)),
    "extraction": index_result["extraction"]["summary"],
    "embedding": index_result["embedding"]["summary"],
    "graph": index_result["graph"]["summary"],
    "catalog": index_result["summary"],
})
assert not index_result["extraction"]["failed_cases"]
assert not index_result["embedding"]["failed_cases"]
assert not index_result["graph"]["failed_cases"]


## 8. Inspect indexing outputs and aggregate usage

`indexed_cases` aliases the newly embedded case records; do not count it a second time. The extraction and embedding sections describe this invocation, while `stored_*` summary counts describe the current catalog.

Timeline and graph embedding totals remain separate because the stages can use different models. Returned totals include reported usage before failures; missing provider usage is not an estimate. These aggregate totals are returned to the caller, not automatically saved as a separate run report.


In [ ]:
show({
    "result_keys": list(index_result),
    "timeline_embedding_usage": index_result["embedding"]["embedding_usage"],
    "timeline_embedding_requests": index_result["embedding"]["embedding_requests"],
    "graph_embedding_usage": index_result["graph"]["embedding_usage"],
    "graph_embedding_requests": index_result["graph"]["embedding_requests"],
    "first_indexed_case": ({
        "id": index_result["indexed_cases"][0]["id"],
        "output": index_result["indexed_cases"][0]["case"].output,
        "entries": len(index_result["indexed_cases"][0]["embeddings"]),
    } if index_result["indexed_cases"] else None),
})


## 9. Retrieve cases using multiple queries

Queries are embedded in batches within this call. Dense and BM25 entry matches are promoted to distinct parent cases. `top_k` applies per query; `max_chars` limits the combined serialized case payloads returned per query.

The report has one `embedding_usage` total and one actual `embedding_requests` count for all supplied queries. Each candidate contains the full `ExtractedCase`. Graph expansion is a separate operation.


In [ ]:
queries = ["expired SSO certificate login failure", "upload worker stale lock", "database timeout missing index"]
retrieval_result = await pipeline.retrieve(
    queries, top_k=3, batch_size=2, max_chars=100_000,
    rpm=300,
)
show({
    "embedding_usage": retrieval_result["embedding_usage"],
    "embedding_requests": retrieval_result["embedding_requests"],
    "results": [
        {
            "query": item["query"], "error": item["error"],
            "used_chars": item["used_chars"], "truncated": item["truncated"],
            "matches": [
                {"id": hit["id"], "product": hit["case"].metadata["product"],
                 "score": round(hit["score"], 4), "item_index": hit["item_index"]}
                for hit in item["candidates"]
            ],
        }
        for item in retrieval_result["results"]
    ],
})
assert all(item["error"] is None for item in retrieval_result["results"])
assert retrieval_result["embedding_requests"] == 2
# Inspect the full case, including review and execution, without changing the retrieval response.
match = retrieval_result["results"][0]["candidates"][0]
show({"id": match["id"], "output": match["case"].output, "review": match["case"].review})


## 10. Expand retrieved case IDs through the saved graph

Expansion makes no model calls. It returns up to `per_case_top_k` neighbors **for each seed**. All supplied seeds are excluded as candidates. Shared neighbors stay in each seed's group; you decide how to merge or deduplicate later. These results contain IDs and weights, not full case records.


In [ ]:
seed_ids = [hit["id"] for hit in retrieval_result["results"][0]["candidates"][:2]]
expansion_result = await pipeline.graph_expansion(seed_ids, per_case_top_k=3)
show(expansion_result)
assert [group["seed_id"] for group in expansion_result] == list(dict.fromkeys(seed_ids))
assert all(len(group["neighbors"]) <= 3 for group in expansion_result)


## 11. Add the remaining cases, then try an unchanged rerun

Incremental indexing extracts and timeline-embeds incoming cases. When the active corpus changes, the local BM25 index and case graph are refreshed; graph summaries currently re-embed the **whole active corpus**. Thus graph request counts can exceed work attributable to only the new cases.

An unchanged rerun skips existing IDs. `embedding` then has zero new requests and empty usage. **Current graph caveat:** `result["graph"]` may be a cached graph result, including usage from its earlier construction. Do not sum that historical graph usage as new spend on every `index()` call.


In [ ]:
update_result = await pipeline.index(incoming_cases)
show({
    "incoming_cases": len(incoming_cases),
    "new_embedding_summary": update_result["embedding"]["summary"],
    "new_embedding_requests": update_result["embedding"]["embedding_requests"],
    "new_embedding_usage": update_result["embedding"]["embedding_usage"],
    "graph_embedding_requests": update_result["graph"]["embedding_requests"],
    "catalog": update_result["summary"],
})
assert not update_result["extraction"]["failed_cases"]
assert not update_result["embedding"]["failed_cases"]
assert not update_result["graph"]["failed_cases"]

unchanged_result = await pipeline.index(selected_cases)
show({
    "skipped_existing": len(unchanged_result["skipped_ids"]),
    "new_embedding_requests": unchanged_result["embedding"]["embedding_requests"],
    "new_embedding_usage": unchanged_result["embedding"]["embedding_usage"],
})
assert unchanged_result["embedding"]["embedding_requests"] == 0


## 12. Replace one case explicitly

Existing IDs are skipped by default even if their source text changed. Use `rewrite=True` to replace that case. The old case's timeline vectors are replaced after successful extraction; corpus-derived indexes refresh as needed.


In [ ]:
changed_case = deepcopy(selected_cases[0])
changed_case["artifacts"][-1]["text"] = "Customer confirmed recovery again after a second validation window."
rewrite_result = await pipeline.index([changed_case], rewrite=True)
show({
    "newly_indexed_ids": [item["id"] for item in rewrite_result["indexed_cases"]],
    "timeline_requests": rewrite_result["embedding"]["embedding_requests"],
    "timeline_usage": rewrite_result["embedding"]["embedding_usage"],
    "stored_embedded": rewrite_result["summary"]["stored_embedded"],
})
assert not rewrite_result["extraction"]["failed_cases"]
assert not rewrite_result["embedding"]["failed_cases"]


## 13. Reopen local storage and retrieve again

A new pipeline instance opens the same catalog and cached indexes. Opening does not re-extract cases. Retrieval still embeds the new query, using the same embedding provider/model as stored timeline vectors.

The catalog contains canonical cases and timeline vectors. `indexes/<revision>/` contains derived BM25 and graph caches. Per-case SQL workspaces are temporary; they are not persistent database files here. We read the catalog below only to inspect storage layout, not as an application API for modifying it.


The final cell also reloads the persisted BM25 index, rebuilds the same lexical corpus explicitly through `BM25Index.from_records`, and compares document IDs. It prints BM25-only matches and the **cosine, BM25, and RRF scores** from hybrid retrieval. Keyword-index construction itself makes no embedding calls.


In [ ]:
from raft.embedding import BM25Index

reopened = open_pipeline(OUTPUT_DIR)
reopened_result = await reopened.retrieve(
    ["expired certificate login"], top_k=2, rpm=300,
)
show({
    "retrieved_ids": [hit["id"] for hit in reopened_result["results"][0]["candidates"]],
    "query_embedding_usage": reopened_result["embedding_usage"],
    "query_embedding_requests": reopened_result["embedding_requests"],
})
assert reopened_result["results"][0]["error"] is None
catalog = json.loads((OUTPUT_DIR / "catalog.json").read_text())
show({
    "catalog_keys": list(catalog),
    "stored_case_count": len(catalog["cases"]),
    "one_record_keys": list(next(iter(catalog["cases"].values()))),
    "files_preview": [str(path.relative_to(OUTPUT_DIR)) for path in sorted(OUTPUT_DIR.rglob("*")) if path.is_file()][:16],
})



bm25_path = OUTPUT_DIR / "indexes" / str(catalog["revision"]) / "bm25"
saved_bm25 = BM25Index.load(bm25_path)
# Explicit standalone lexical build; the pipeline already did this automatically.
all_entry_records = [
    entry
    for record in catalog["cases"].values()
    if record["status"] == "embedded"
    for entry in record["embedding"]["embeddings"]
]
rebuilt_bm25 = BM25Index.from_records(all_entry_records)
assert saved_bm25.documents == rebuilt_bm25.documents
assert all(
    hit["bm25_score"] is not None
    for hit in reopened_result["results"][0]["candidates"]
)
show({
    "saved_bm25_directory": str(bm25_path.relative_to(REPO)),
    "bm25_document_count": len(saved_bm25.documents),
    "lexical_only_matches": [
        {"case_id": hit["case_id"], "item_index": hit["item_index"],
         "bm25_score": round(hit["score"], 4), "text": hit["text"]}
        for hit in saved_bm25.search("expired certificate login", k=3)
    ],
    "hybrid_matches": [
        {"id": hit["id"], "cosine_similarity": round(hit["cosine_similarity"], 4),
         "bm25_score": round(hit["bm25_score"], 4), "rrf_score": round(hit["score"], 4)}
        for hit in reopened_result["results"][0]["candidates"]
    ],
})


## 14. What to change for your own data

- Replace input JSON and field mappings; preserve stable case IDs for updates.
- Supply your Pydantic state/review schemas, worker/reviewer instructions, tools, and models.
- Supply `state_to_text` for entry embeddings and `case_to_text` for graph summaries.
- Set case concurrency, agent concurrency/RPM, and each embedding stage's `batch_size` independently.
- Set `SHOW_PROGRESS=True` for stage bars; install the notebook extra for widget support.
- Choose what execution details to save externally. Per-case agent usage and operation-level embedding usage have different scopes.

The runnable cells deliberately expose each result variable: `extraction_result`, `embedding_result`, `graph_result`, `index_result`, `retrieval_result`, and `expansion_result`. Inspect them directly to see fields omitted by the compact previews.

References: the repository's `examples/pipeline.py`, `src/raft/pipeline.py`, and `src/raft/tools.py` define this walkthrough's public contracts. For live embeddings, see the [official OpenAI embedding model documentation](https://developers.openai.com/api/docs/models/text-embedding-3-small). Saved cell outputs show real API results from this walkthrough.


In [ ]:
for client in clients:
    await client.close()
print("Walkthrough complete. Local files remain at:", OUTPUT_DIR.relative_to(REPO))
